In [ ]:
# %%
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix,
    top_k_accuracy_score,
    f1_score
)
import optuna
from optuna.samplers import TPESampler
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import multiprocessing
warnings.filterwarnings('ignore')

# Get number of CPU cores
N_CORES = multiprocessing.cpu_count()
print(f"Detected {N_CORES} CPU cores")

print("="*80)
print("XGBOOST DESTINATION PREDICTION MODEL TRAINING (OPTIMIZED)")
print("="*80)

# %%
# =====================================================
# STEP 1: Load Data
# =====================================================
print("\n[1/10] Loading data...")
X_features = pd.read_csv('X_features.csv')
y_target = pd.read_csv('y_target.csv')
complete_trips = pd.read_csv('complete_trips_with_features.csv')

# Get the destination column name
y_target = y_target.iloc[:, 0]  # First column

print(f"✓ Loaded {len(X_features)} samples")
print(f"✓ Original number of destination classes: {y_target.nunique()}")

# %%
# =====================================================
# STEP 2: Handle Class Imbalance - Top N + OTHER
# =====================================================
print("\n[2/10] Handling class imbalance with Top-N strategy...")

# Configuration
TOP_N_DESTINATIONS = 75  # You can adjust this (50-100)

# Count destination frequencies
destination_counts = y_target.value_counts()
print(f"\nDestination distribution:")
print(f"  Total unique destinations: {len(destination_counts)}")
print(f"  Most common destination: {destination_counts.iloc[0]} trips")
print(f"  Least common destination: {destination_counts.iloc[-1]} trips")
print(f"  Median trips per destination: {destination_counts.median():.0f}")

# Get top N destinations
top_destinations = destination_counts.head(TOP_N_DESTINATIONS).index.tolist()

# Create new target with "OTHER" category
y_target_categorized = y_target.copy()
y_target_categorized = y_target_categorized.apply(
    lambda x: x if x in top_destinations else 'OTHER'
)

# Statistics
n_other = (y_target_categorized == 'OTHER').sum()
print(f"\n✓ Kept top {TOP_N_DESTINATIONS} destinations")
print(f"✓ Moved {n_other} trips ({n_other/len(y_target)*100:.1f}%) to 'OTHER' category")
print(f"✓ Final number of classes: {y_target_categorized.nunique()}")

# Update target
y_target = y_target_categorized

# %%
# =====================================================
# STEP 3: Time-based Train/Val/Test Split
# =====================================================
print("\n[3/10] Creating time-based splits...")

# Get timestamps for splitting
timestamps = pd.to_datetime(complete_trips['pickup_timestamp'])

# Sort by time
sort_idx = timestamps.argsort()
X_features_sorted = X_features.iloc[sort_idx].reset_index(drop=True)
y_target_sorted = y_target.iloc[sort_idx].reset_index(drop=True)
timestamps_sorted = timestamps.iloc[sort_idx].reset_index(drop=True)

# Calculate split indices
n_samples = len(X_features_sorted)
train_end = int(n_samples * 0.70)
val_end = int(n_samples * 0.85)

# Split data
X_train = X_features_sorted.iloc[:train_end].copy()
y_train = y_target_sorted.iloc[:train_end].copy()

X_val = X_features_sorted.iloc[train_end:val_end].copy()
y_val = y_target_sorted.iloc[train_end:val_end].copy()

X_test = X_features_sorted.iloc[val_end:].copy()
y_test = y_target_sorted.iloc[val_end:].copy()

print(f"\nTime-based split:")
print(f"  Train: {len(X_train)} samples ({len(X_train)/n_samples*100:.1f}%)")
print(f"    Date range: {timestamps_sorted.iloc[0]} to {timestamps_sorted.iloc[train_end-1]}")
print(f"  Validation: {len(X_val)} samples ({len(X_val)/n_samples*100:.1f}%)")
print(f"    Date range: {timestamps_sorted.iloc[train_end]} to {timestamps_sorted.iloc[val_end-1]}")
print(f"  Test: {len(X_test)} samples ({len(X_test)/n_samples*100:.1f}%)")
print(f"    Date range: {timestamps_sorted.iloc[val_end]} to {timestamps_sorted.iloc[-1]}")

# %%
# =====================================================
# STEP 4: Label Encoding for Categorical Features
# =====================================================
print("\n[4/10] Encoding categorical features...")

# Identify categorical columns
categorical_cols = ['origin_h3', 'time_period']

# Initialize label encoders
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    
    # Fit on training data only
    le.fit(X_train[col].astype(str))
    
    # Transform all splits
    X_train[col] = le.transform(X_train[col].astype(str))
    
    # Handle unseen categories in val/test
    X_val[col] = X_val[col].astype(str).apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    X_test[col] = X_test[col].astype(str).apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    
    label_encoders[col] = le
    print(f"✓ Encoded '{col}': {len(le.classes_)} unique values")

# Encode target variable
target_encoder = LabelEncoder()
y_train_encoded = target_encoder.fit_transform(y_train)
y_val_encoded = target_encoder.transform(y_val)
y_test_encoded = target_encoder.transform(y_test)

print(f"✓ Encoded target: {len(target_encoder.classes_)} classes")

# %%
# =====================================================
# STEP 5: Prepare XGBoost Data Structures
# =====================================================
print("\n[5/10] Preparing XGBoost datasets...")

dtrain = xgb.DMatrix(X_train, label=y_train_encoded)
dval = xgb.DMatrix(X_val, label=y_val_encoded)
dtest = xgb.DMatrix(X_test, label=y_test_encoded)

print("✓ XGBoost DMatrix objects created")

# %%
# =====================================================
# STEP 6: Baseline Model (Before Tuning)
# =====================================================
print("\n[6/10] Training baseline model...")

baseline_params = {
    'objective': 'multi:softprob',
    'num_class': len(target_encoder.classes_),
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cpu',
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'seed': 42,
    'nthread': N_CORES  # USE ALL CORES
}

print(f"Training baseline model with {N_CORES} cores...")
baseline_model = xgb.train(
    baseline_params,
    dtrain,
    num_boost_round=100,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=10,
    verbose_eval=False
)

# Baseline predictions
y_val_pred_baseline = baseline_model.predict(dval)
y_val_pred_classes_baseline = np.argmax(y_val_pred_baseline, axis=1)

baseline_accuracy = accuracy_score(y_val_encoded, y_val_pred_classes_baseline)
print(f"\n✓ Baseline validation accuracy: {baseline_accuracy:.4f}")

# %%
# =====================================================
# STEP 7: Hyperparameter Tuning with Optuna (PARALLEL)
# =====================================================
print("\n[7/10] Starting Optuna hyperparameter tuning (PARALLEL)...")
print(f"Running with {N_CORES} parallel jobs - this should be MUCH faster!")

def objective(trial):
    """Optuna objective function for XGBoost hyperparameter tuning"""
    
    param = {
        'objective': 'multi:softprob',
        'num_class': len(target_encoder.classes_),
        'eval_metric': 'mlogloss',
        'tree_method': 'hist',
        'device': 'cpu',
        'seed': 42,
        'nthread': max(1, N_CORES // 4),  # Divide cores among parallel trials
        
        # Hyperparameters to tune
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
    }
    
    # Train model
    model = xgb.train(
        param,
        dtrain,
        num_boost_round=200,
        evals=[(dval, 'val')],
        early_stopping_rounds=15,
        verbose_eval=False
    )
    
    # Predict and evaluate
    y_pred = model.predict(dval)
    y_pred_classes = np.argmax(y_pred, axis=1)
    accuracy = accuracy_score(y_val_encoded, y_pred_classes)
    
    return accuracy

# Create Optuna study with parallel execution
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42)
)

# Run optimization WITH PARALLEL JOBS
n_jobs = max(1, N_CORES // 2)  # Use half the cores for parallel trials
print(f"\nRunning 75 trials with {n_jobs} parallel jobs...")
print("⚡ PARALLEL MODE ACTIVATED - Expect 2-4x speedup!")

study.optimize(
    objective, 
    n_trials=75,
    show_progress_bar=True,
    n_jobs=n_jobs  # PARALLEL EXECUTION
)

print("\n✓ Hyperparameter tuning complete!")
print(f"\nBest validation accuracy: {study.best_value:.4f}")
print(f"Improvement over baseline: {(study.best_value - baseline_accuracy)*100:.2f}%")
print("\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# %%
# =====================================================
# STEP 8: Train Final Model with Best Parameters
# =====================================================
print("\n[8/10] Training final model with best parameters...")

best_params = {
    'objective': 'multi:softprob',
    'num_class': len(target_encoder.classes_),
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cpu',
    'seed': 42,
    'nthread': N_CORES,  # USE ALL CORES FOR FINAL MODEL
    **study.best_params
}

print(f"Training with {N_CORES} cores...")
final_model = xgb.train(
    best_params,
    dtrain,
    num_boost_round=300,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=20,
    verbose_eval=False
)

print("✓ Final model trained!")

# %%
# =====================================================
# STEP 9: Comprehensive Model Evaluation
# =====================================================
print("\n[9/10] Evaluating final model...")

# Predictions on all sets
y_train_pred = final_model.predict(dtrain)
y_train_pred_classes = np.argmax(y_train_pred, axis=1)

y_val_pred = final_model.predict(dval)
y_val_pred_classes = np.argmax(y_val_pred, axis=1)

y_test_pred = final_model.predict(dtest)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)

# --- Accuracy Metrics ---
print("\n" + "="*80)
print("ACCURACY METRICS")
print("="*80)

train_acc = accuracy_score(y_train_encoded, y_train_pred_classes)
val_acc = accuracy_score(y_val_encoded, y_val_pred_classes)
test_acc = accuracy_score(y_test_encoded, y_test_pred_classes)

print(f"Train Accuracy:      {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy:       {test_acc:.4f}")

# --- Top-K Accuracy ---
print("\n" + "="*80)
print("TOP-K ACCURACY (Is correct destination in top K predictions?)")
print("="*80)

for k in [1, 3, 5, 10]:
    if k <= len(target_encoder.classes_):
        top_k_train = top_k_accuracy_score(y_train_encoded, y_train_pred, k=k)
        top_k_val = top_k_accuracy_score(y_val_encoded, y_val_pred, k=k)
        top_k_test = top_k_accuracy_score(y_test_encoded, y_test_pred, k=k)
        
        print(f"\nTop-{k} Accuracy:")
        print(f"  Train:      {top_k_train:.4f}")
        print(f"  Validation: {top_k_val:.4f}")
        print(f"  Test:       {top_k_test:.4f}")

# --- F1 Scores ---
print("\n" + "="*80)
print("F1 SCORES")
print("="*80)

f1_macro_test = f1_score(y_test_encoded, y_test_pred_classes, average='macro')
f1_weighted_test = f1_score(y_test_encoded, y_test_pred_classes, average='weighted')

print(f"Test F1 (Macro):    {f1_macro_test:.4f}")
print(f"Test F1 (Weighted): {f1_weighted_test:.4f}")

# --- Classification Report ---
print("\n" + "="*80)
print("DETAILED CLASSIFICATION REPORT (Test Set)")
print("="*80)

# Get class names
class_names = target_encoder.classes_

# Generate report
report = classification_report(
    y_test_encoded, 
    y_test_pred_classes,
    target_names=[str(c) for c in class_names],
    digits=3,
    zero_division=0
)
print(report)

# %%
# =====================================================
# STEP 10: Feature Importance Analysis
# =====================================================
print("\n[10/10] Analyzing feature importance...")

# Get feature importance
importance_dict = final_model.get_score(importance_type='gain')

# Convert to DataFrame
feature_importance = pd.DataFrame({
    'feature': list(importance_dict.keys()),
    'importance': list(importance_dict.values())
}).sort_values('importance', ascending=False)

# Map feature indices back to names
feature_names = X_train.columns.tolist()
feature_importance['feature_name'] = feature_importance['feature'].apply(
    lambda x: feature_names[int(x.replace('f', ''))] if 'f' in str(x) else x
)

# Plot feature importance
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature_name'])
plt.xlabel('Importance (Gain)')
plt.title('Top 15 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
print("\n✓ Feature importance plot saved: feature_importance.png")

# %%
# =====================================================
# Confusion Matrix (Top 10 Classes)
# =====================================================
print("\nGenerating confusion matrix for top destinations...")

# Get top 10 most common destinations in test set
top_10_classes = pd.Series(y_test_encoded).value_counts().head(10).index.tolist()

# Filter predictions for top 10
mask = np.isin(y_test_encoded, top_10_classes)
y_test_top10 = y_test_encoded[mask]
y_pred_top10 = y_test_pred_classes[mask]

# Create confusion matrix
cm = confusion_matrix(y_test_top10, y_pred_top10, labels=top_10_classes)

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=[class_names[i][:15] for i in top_10_classes],
    yticklabels=[class_names[i][:15] for i in top_10_classes]
)
plt.xlabel('Predicted Destination')
plt.ylabel('True Destination')
plt.title('Confusion Matrix - Top 10 Destinations')
plt.tight_layout()
plt.savefig('confusion_matrix_top10.png', dpi=300, bbox_inches='tight')
print("✓ Confusion matrix saved: confusion_matrix_top10.png")

# %%
# =====================================================
# Save Model and Encoders
# =====================================================
print("\nSaving model and encoders...")

# Create timestamp for versioning
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save XGBoost model
model_filename = f'xgboost_destination_model_{timestamp}.json'
final_model.save_model(model_filename)
print(f"✓ Model saved: {model_filename}")

# Save encoders
encoders_dict = {
    'label_encoders': label_encoders,
    'target_encoder': target_encoder,
    'feature_names': X_train.columns.tolist(),
    'top_destinations': top_destinations,
    'best_params': best_params,
    'metrics': {
        'test_accuracy': test_acc,
        'test_f1_macro': f1_macro_test,
        'test_f1_weighted': f1_weighted_test,
        'top_3_accuracy': top_k_accuracy_score(y_test_encoded, y_test_pred, k=3),
        'top_5_accuracy': top_k_accuracy_score(y_test_encoded, y_test_pred, k=5)
    }
}

encoders_filename = f'model_encoders_{timestamp}.pkl'
joblib.dump(encoders_dict, encoders_filename)
print(f"✓ Encoders and metadata saved: {encoders_filename}")

# Save feature importance
feature_importance.to_csv(f'feature_importance_{timestamp}.csv', index=False)
print(f"✓ Feature importance saved: feature_importance_{timestamp}.csv")

# Save Optuna study
study_filename = f'optuna_study_{timestamp}.pkl'
joblib.dump(study, study_filename)
print(f"✓ Optuna study saved: {study_filename}")

# %%
# =====================================================
# FINAL SUMMARY
# =====================================================
print("\n" + "="*80)
print("TRAINING COMPLETE! 🎉")
print("="*80)
print("\nModel Performance Summary:")
print(f"  Test Accuracy:        {test_acc:.4f}")
print(f"  Test Top-3 Accuracy:  {top_k_accuracy_score(y_test_encoded, y_test_pred, k=3):.4f}")
print(f"  Test Top-5 Accuracy:  {top_k_accuracy_score(y_test_encoded, y_test_pred, k=5):.4f}")
print(f"  Test F1 (Weighted):   {f1_weighted_test:.4f}")

print("\nFiles Generated:")
print(f"  1. {model_filename}")
print(f"  2. {encoders_filename}")
print(f"  3. feature_importance.png")
print(f"  4. confusion_matrix_top10.png")
print(f"  5. feature_importance_{timestamp}.csv")
print(f"  6. {study_filename}")

print("\n" + "="*80)
print("Performance Boost Info:")
print(f"  CPU Cores Used: {N_CORES}")
print(f"  Optuna Parallel Jobs: {n_jobs}")
print("  Expected Speedup: 2-4x faster than original!")
print("="*80)

Detected 8 CPU cores
XGBOOST DESTINATION PREDICTION MODEL TRAINING (OPTIMIZED)

[1/10] Loading data...
✓ Loaded 336028 samples
✓ Original number of destination classes: 330

[2/10] Handling class imbalance with Top-N strategy...

Destination distribution:
  Total unique destinations: 330
  Most common destination: 8334 trips
  Least common destination: 100 trips
  Median trips per destination: 505

✓ Kept top 75 destinations
✓ Moved 118109 trips (35.1%) to 'OTHER' category
✓ Final number of classes: 76

[3/10] Creating time-based splits...

Time-based split:
  Train: 235219 samples (70.0%)
    Date range: 2024-11-01 00:01:10 to 2024-11-22 02:26:03
  Validation: 50404 samples (15.0%)
    Date range: 2024-11-22 02:27:13 to 2024-11-26 15:24:30
  Test: 50405 samples (15.0%)
    Date range: 2024-11-26 15:24:32 to 2024-11-30 23:57:38

[4/10] Encoding categorical features...
✓ Encoded 'origin_h3': 917 unique values
✓ Encoded 'time_period': 4 unique values
✓ Encoded target: 76 classes

[5/10] 

[I 2025-10-02 09:52:11,532] A new study created in memory with name: no-name-8e7f21c6-4e2a-4c5a-be10-1ba5327c4dc2



✓ Baseline validation accuracy: 0.4701

[7/10] Starting Optuna hyperparameter tuning (PARALLEL)...
Running with 8 parallel jobs - this should be MUCH faster!

Running 75 trials with 4 parallel jobs...
⚡ PARALLEL MODE ACTIVATED - Expect 2-4x speedup!


Best trial: 3. Best value: 0.47032:   1%|▏         | 1/75 [02:33<3:09:49, 153.92s/it]

[I 2025-10-02 09:54:45,464] Trial 3 finished with value: 0.47031981588762795 and parameters: {'max_depth': 4, 'learning_rate': 0.10226063041083484, 'min_child_weight': 7, 'subsample': 0.997812661248205, 'colsample_bytree': 0.7337961514476261, 'gamma': 1.730709038833171, 'reg_alpha': 2.3498731764868013, 'reg_lambda': 3.7684563240212468}. Best is trial 3 with value: 0.47031981588762795.


...

[I 2025-10-02 11:07:37,120] Trial 73 finished with value: 0.46982382350607094 and parameters: {'max_depth': 5, 'learning_rate': 0.01153968339693766, 'min_child_weight': 6, 'subsample': 0.6593135309539936, 'colsample_bytree': 0.7276040671859547, 'gamma': 1.2650936038776999, 'reg_alpha': 0.24631326564020695, 'reg_lambda': 1.0386771076195511}. Best is trial 58 with value: 0.47067693040234904.


Best trial: 58. Best value: 0.470677: 100%|██████████| 75/75 [1:15:40<00:00, 60.54s/it]


[I 2025-10-02 11:07:52,364] Trial 74 finished with value: 0.4703594952781525 and parameters: {'max_depth': 6, 'learning_rate': 0.011426020229069083, 'min_child_weight': 6, 'subsample': 0.6726393766248767, 'colsample_bytree': 0.7912277512113659, 'gamma': 0.5616726218094418, 'reg_alpha': 1.2334732812983333, 'reg_lambda': 1.39866826547315}. Best is trial 58 with value: 0.47067693040234904.

✓ Hyperparameter tuning complete!

Best validation accuracy: 0.4707
Improvement over baseline: 0.05%

Best hyperparameters:
  max_depth: 6
  learning_rate: 0.018578546856500594
  min_child_weight: 5
  subsample: 0.781323986880641
  colsample_bytree: 0.6958633417937001
  gamma: 1.433712578032936
  reg_alpha: 0.931178481165737
  reg_lambda: 1.8529785956665232

[8/10] Training final model with best parameters...
Training with 8 cores...
✓ Final model trained!

[9/10] Evaluating final model...

ACCURACY METRICS
Train Accuracy:      0.4881
Validation Accuracy: 0.4707
Test Accuracy:       0.4785

TOP-K ACCUR

ValueError: invalid literal for int() with base 10: 'zone_hour_avg_are'

: 